Smart Trash Bin - YOLOv5

In [ ]:
!git clone https://github.com/ultralytics/yolov5.git
%cd yolov5
!pip install -r requirements.txt --quiet

Dataset downloaded using Roboflow

In [ ]:
!python train.py \
--img 640 \
--batch 8 \
--epochs 30 \
--data data.yaml \
--weights yolov5s.pt \
--name smart_trash

In [ ]:
!python detect.py \
--weights /content/drive/MyDrive/best_boost.pt \
--img 832 \
--conf 0.2 \
--iou 0.4 \
--source /content/videos \
--project runs/detect \
--name final_better \
--save-txt \
--save-conf \
--exist-ok

In [ ]:
import cv2
import torch

model = torch.hub.load('ultralytics/yolov5', 'custom',
                       path='/content/drive/MyDrive/best_boost.pt',
                       force_reload=True)

video_path = "/content/videos/YOUR_VIDEO.mp4"
cap = cv2.VideoCapture(video_path)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter("tracked_output.mp4", fourcc, 20.0,
                      (int(cap.get(3)), int(cap.get(4))))

prev_boxes = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    boxes = results.xyxy[0].cpu().numpy()

    smoothed_boxes = []
    for b in boxes:
        x1, y1, x2, y2, conf, cls = b
        if conf > 0.2:
            smoothed_boxes.append([x1, y1, x2, y2])

    for box in smoothed_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    out.write(frame)

cap.release()
out.release()